# Zimbabwe Student Dropout — Model Training

Trains **Logistic Regression**, **KNN**, and **Naive Bayes** on the Zimbabwe
student dropout dataset, then saves the fitted pipelines + schema + metrics
to `artifacts/` for the Streamlit app.

Click *Run All* in VS Code's Jupyter toolbar.

In [ ]:
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = r"C:/Users/nicky/Documents/zimbabwe_student_dropout (1).csv"
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Load data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

## 2. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df['dropout'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#4caf50', '#f44336']
)
axes[0].set_title('Dropout class distribution')
axes[0].set_xticklabels(['Stayed (0)', 'Dropped (1)'], rotation=0)
axes[0].set_ylabel('Count')

sns.heatmap(df.isna(), cbar=False, ax=axes[1])
axes[1].set_title('Missingness pattern')
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include='number').columns
plt.figure(figsize=(9, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numeric feature correlation')
plt.tight_layout()
plt.show()

## 3. Clean & prepare features

- `fees_paid` NaN → treat as a third category `"None"` (column otherwise contains only Full / Partial).
- `previous_grade` is ordinal on the Zim scale: `F < 3 < 2.2 < 2.1 < 1` — map to integers.
- Numeric NaNs in `family_income`, `study_hours`, `attendance_rate` are handled by `SimpleImputer(median)` inside the pipeline.

In [ ]:
data = df.copy()
data['fees_paid'] = data['fees_paid'].fillna('None')

grade_map = {'F': 0, '3': 1, '2.2': 2, '2.1': 3, '1': 4}
data['previous_grade_ord'] = data['previous_grade'].map(grade_map)
data = data.drop(columns=['previous_grade'])

print("Cleaned dtypes:")
print(data.dtypes)
print("\nfees_paid unique:", sorted(data['fees_paid'].unique()))
print("previous_grade_ord unique:", sorted(data['previous_grade_ord'].dropna().unique()))

## 4. Train / test split (stratified, 80 / 20)

In [ ]:
X = data.drop(columns=['dropout'])
y = data['dropout']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
print(f"Train: {X_train.shape}    Test: {X_test.shape}")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  class balance: {y_test.value_counts(normalize=True).round(3).to_dict()}")

## 5. Preprocessing pipeline

- **Numeric**: median impute → standard scale (needed for LogReg and KNN).
- **Categorical**: most-frequent impute → one-hot encode (handles unseen labels).

In [ ]:
NUMERIC_COLS = ['age', 'family_income', 'transport_time', 'study_hours',
                'attendance_rate', 'lms_logins', 'previous_grade_ord', 'stress_level']
CATEGORICAL_COLS = ['gender', 'location', 'internet_access',
                    'electricity_reliability', 'fees_paid', 'part_time_job']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale',  StandardScaler())]), NUMERIC_COLS),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))]),
     CATEGORICAL_COLS),
])
preprocessor

## 6. Train three models

In [ ]:
def evaluate(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    m = {
        'accuracy':  float(accuracy_score(y_eval, pred)),
        'precision': float(precision_score(y_eval, pred, zero_division=0)),
        'recall':    float(recall_score(y_eval, pred, zero_division=0)),
        'f1':        float(f1_score(y_eval, pred, zero_division=0)),
    }
    print(f"\n=== {name} ===")
    for k, v in m.items():
        print(f"  {k:<10} {v:.4f}")
    print(classification_report(y_eval, pred, zero_division=0,
                                target_names=['Stayed', 'Dropped']))
    return m

results = {}

### 6a. Logistic Regression — full training set, balanced class weights

Without `class_weight='balanced'` the model collapses to the majority class
(predicts "Dropped" for everyone) because the features are weak predictors
and the 65/35 class imbalance dominates the loss. Balancing forces the model
to weight both classes equally.

In [ ]:
print("Training Logistic Regression (class_weight='balanced')...")
t0 = time.time()
logreg = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                               class_weight='balanced')),
])
logreg.fit(X_train, y_train)
print(f"  fit in {time.time() - t0:.1f}s")

results['Logistic Regression'] = evaluate('Logistic Regression', logreg, X_test, y_test)

### 6b. K-Nearest Neighbours — 50,000-row subsample

KNN stores every training point and computes distances at predict time. With
the full 419k training rows the ball-tree predict step takes well over an hour
for a meaningful test slice, so we train KNN on a stratified 50k subsample
(standard practice for KNN on hundred-thousand-row tabular data).
Prediction is then evaluated on a 20k random test subset.

In [ ]:
KNN_TRAIN_SAMPLE = 50_000
KNN_EVAL_SAMPLE = 20_000

rng = np.random.RandomState(RANDOM_STATE)
train_idx = rng.choice(len(X_train), size=KNN_TRAIN_SAMPLE, replace=False)
X_knn = X_train.iloc[train_idx]
y_knn = y_train.iloc[train_idx]

print(f"Training KNN (k=15) on {KNN_TRAIN_SAMPLE} rows...")
t0 = time.time()
knn = Pipeline([
    ('pre', preprocessor),
    ('clf', KNeighborsClassifier(n_neighbors=15, n_jobs=-1)),
])
knn.fit(X_knn, y_knn)
print(f"  fit in {time.time() - t0:.1f}s")

eval_idx = rng.choice(len(X_test), size=KNN_EVAL_SAMPLE, replace=False)
X_test_s = X_test.iloc[eval_idx]
y_test_s = y_test.iloc[eval_idx]

print(f"Predicting KNN on {KNN_EVAL_SAMPLE} test rows...")
t0 = time.time()
results['KNN'] = evaluate('KNN', knn, X_test_s, y_test_s)
print(f"  predict in {time.time() - t0:.1f}s")

### 6c. Gaussian Naive Bayes — full training set, balanced priors

Setting `priors=[0.5, 0.5]` overrides the dataset's 65/35 class prior so the
model doesn't collapse to predicting "Dropped" for every input.

In [ ]:
print("Training Naive Bayes (priors=[0.5, 0.5])...")
t0 = time.time()
nb = Pipeline([
    ('pre', preprocessor),
    ('clf', GaussianNB(priors=[0.5, 0.5])),
])
nb.fit(X_train, y_train)
print(f"  fit in {time.time() - t0:.1f}s")

results['Naive Bayes'] = evaluate('Naive Bayes', nb, X_test, y_test)

## 7. Compare models

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df.round(4))

ax = results_df.plot(kind='bar', figsize=(10, 5), rot=0,
                     color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax.set_title('Model comparison on test set')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, [
    ('Logistic Regression', logreg),
    ('KNN', knn),
    ('Naive Bayes', nb),
]):
    if name == 'KNN':
        cm = confusion_matrix(y_test_s, model.predict(X_test_s))
    else:
        cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stayed', 'Dropped'],
                yticklabels=['Stayed', 'Dropped'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. Save artifacts

In [ ]:
joblib.dump(logreg, ARTIFACTS / 'logreg.joblib')
joblib.dump(knn,    ARTIFACTS / 'knn.joblib')
joblib.dump(nb,     ARTIFACTS / 'nb.joblib')

schema = {
    'numeric': {}, 'categorical': {},
    'grade_map': grade_map,
    'numeric_order': NUMERIC_COLS,
    'categorical_order': CATEGORICAL_COLS,
    'grade_labels': ['1', '2.1', '2.2', '3', 'F'],
}
for c in NUMERIC_COLS:
    if c == 'previous_grade_ord':
        continue
    s = data[c].dropna()
    schema['numeric'][c] = {'min': float(s.min()), 'max': float(s.max()),
                            'median': float(s.median())}
for c in CATEGORICAL_COLS:
    schema['categorical'][c] = sorted(data[c].dropna().unique().tolist())

with open(ARTIFACTS / 'feature_columns.json', 'w') as f:
    json.dump(schema, f, indent=2)

with open(ARTIFACTS / 'metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to artifacts/:")
for p in sorted(ARTIFACTS.iterdir()):
    print(f"  {p.name:<25} {p.stat().st_size / 1024:>10.1f} KB")

## 9. Sanity check — reload models and predict on a sample

In [ ]:
sample_raw = X_test.iloc[[0]].copy()
print("Input row:")
print(sample_raw.T)
print(f"\nActual label: {int(y_test.iloc[0])}  ({'Dropped' if y_test.iloc[0] == 1 else 'Stayed'})\n")

for name, fname in [
    ('Logistic Regression', 'logreg.joblib'),
    ('KNN',                 'knn.joblib'),
    ('Naive Bayes',         'nb.joblib'),
]:
    m = joblib.load(ARTIFACTS / fname)
    pred = int(m.predict(sample_raw)[0])
    proba = float(m.predict_proba(sample_raw)[0, 1])
    label = 'Dropped' if pred == 1 else 'Stayed'
    print(f"  {name:<22} -> {label:<8}  (P(dropout) = {proba:.3f})")